In [1]:
import psycopg2 
import sys
import pandas as pd
cnx = psycopg2.connect(
            host='10.1.3.102',
            database='postgres',
            user='printer_data',
            password='printer_data',
            port=5432
        )
 
query = '''
SELECT 
    时间::DATE AS error_date,   -- Converts timestamp to YYYY-MM-DD
    printer_id, 
    内容, 
    COUNT(*) AS occurance
FROM 
    public.printer_almlist
GROUP BY 
    时间::DATE,   -- You must group by the converted date
    printer_id, 
    内容
ORDER BY 
    error_date DESC, 
    occurance DESC;
'''

try:    
    cursor = cnx.cursor()
    cursor.execute(query)
    # Fetch the data
    rows = cursor.fetchall()
   
    # Get column names from cursor.description
    columns = [desc[0] for desc in cursor.description]
   
    # Convert to DataFrame with column names
    df_log = pd.DataFrame(rows, columns=columns)
    cursor.close()
    print("✅ Fetched printer summary data from PostgreSQL")
except Exception as e:  
    print(f"Error fetching printer data: {e}")
    sys.exit(1)

print(df_log.head(5))  

✅ Fetched printer summary data from PostgreSQL
   error_date        printer_id  \
0  2025-12-25  SPT1.2_printer02   
1  2025-12-25  SPT1.2_printer02   
2  2025-12-25  SPT1.2_printer02   
3  2025-12-25  SPT1.2_printer04   
4  2025-12-25  SPT1.2_printer04   

                                                  内容  occurance  
0  扫描PLC更新界面出现异常：Input array is longer than the n...       5704  
1  自动运行后台刷新出现异常：Cannot find table 0.,    at Syste...       5268  
2  扫描PLC更新界面出现异常：Cannot find table 0.,    at Syst...       1773  
3                       :,568Mesh plate not detected        680  
4  :,527Downstream device network communication f...        604  


In [2]:
# Extract 3-digit error codes from 内容 like " :,568..." and filter rows without a code
pattern = r'[:：],\s*(\d{3})'

df_log_with_code = df_log.copy()
df_log_with_code['error_code'] = df_log_with_code['内容'].str.extract(pattern, expand=False)

# Keep only rows that contain an error_code
df_log_errors = df_log_with_code.dropna(subset=['error_code']).copy()
df_log_errors['error_code'] = df_log_errors['error_code'].astype(int)

# Keep original columns and rename error_date -> date
df = df_log_errors[['error_date', 'printer_id', 'error_code', '内容', 'occurance']].copy()
df.rename(columns={'error_date': 'date'}, inplace=True)

# Remove embedded error_code from 内容 and trim
df['内容'] = df['内容'].str.replace(pattern, '', regex=True).str.strip()

print(df.head(10))


          date        printer_id  error_code  \
3   2025-12-25  SPT1.2_printer04         568   
4   2025-12-25  SPT1.2_printer04         527   
5   2025-12-25  SPT1.1_printer01         589   
6   2025-12-25  SPT1.2_printer01         523   
7   2025-12-25  SPT1.1_printer04         527   
8   2025-12-25  SPT1.2_printer04         337   
9   2025-12-25  SPT1.2_printer02         566   
10  2025-12-25  SPT1.1_printer01         248   
11  2025-12-25  SPT1.2_printer02         248   
12  2025-12-25  SPT1.2_printer02         563   

                                                   内容  occurance  
3                             Mesh plate not detected        680  
4     Downstream device network communication failure        604  
5                                                            493  
6   Shielding the camera data status of the outgoi...        337  
7                                          下游设备网络通讯失败        322  
8   Abnormal slice is found in the jacking cylinde...        226  
9 

In [8]:
# Split df into top-4 printers for SPT1.1 and SPT1.2 separately

# Filter by line (IMPORTANT: filter using df['printer_id'], not df_log, to avoid index mismatch)
df_11 = df[df['printer_id'].str.startswith('SPT1.1_')].copy()
df_12 = df[df['printer_id'].str.startswith('SPT1.2_')].copy()

# Top 4 printer_ids in each line by TOTAL occurrences (not just number of rows)
top4_ids_11 = df_11.groupby('printer_id')['occurance'].sum().nlargest(4).index.tolist()
top4_ids_12 = df_12.groupby('printer_id')['occurance'].sum().nlargest(4).index.tolist()

# Build dicts for quick access
dfs_by_printer_11 = {pid: df_11.loc[df_11['printer_id'] == pid].copy() for pid in top4_ids_11}
dfs_by_printer_12 = {pid: df_12.loc[df_12['printer_id'] == pid].copy() for pid in top4_ids_12}

# Helper to safely get nth dataframe (pads with empty if missing)
def _get_df(dfs_dict, ids, idx):
    return dfs_dict[ids[idx]] if len(ids) > idx else pd.DataFrame(columns=df.columns)

# Create 4 DataFrames for SPT1.1
df_spt11_printer_1 = _get_df(dfs_by_printer_11, top4_ids_11, 0)
df_spt11_printer_2 = _get_df(dfs_by_printer_11, top4_ids_11, 1)
df_spt11_printer_3 = _get_df(dfs_by_printer_11, top4_ids_11, 2)
df_spt11_printer_4 = _get_df(dfs_by_printer_11, top4_ids_11, 3)

# Create 4 DataFrames for SPT1.2
df_spt12_printer_1 = _get_df(dfs_by_printer_12, top4_ids_12, 0)
df_spt12_printer_2 = _get_df(dfs_by_printer_12, top4_ids_12, 1)
df_spt12_printer_3 = _get_df(dfs_by_printer_12, top4_ids_12, 2)
df_spt12_printer_4 = _get_df(dfs_by_printer_12, top4_ids_12, 3)

print("SPT1.1 top 4 printers:", top4_ids_11)
for i, pid in enumerate(top4_ids_11, 1):
    print(f"df_spt11_printer_{i}: {pid}, shape={dfs_by_printer_11[pid].shape}")

print("SPT1.2 top 4 printers:", top4_ids_12)
for i, pid in enumerate(top4_ids_12, 1):
    print(f"df_spt12_printer_{i}: {pid}, shape={dfs_by_printer_12[pid].shape}")

SPT1.1 top 4 printers: ['SPT1.1_printer04', 'SPT1.1_printer02', 'SPT1.1_printer03', 'SPT1.1_printer01']
df_spt11_printer_1: SPT1.1_printer04, shape=(6685, 5)
df_spt11_printer_2: SPT1.1_printer02, shape=(7315, 5)
df_spt11_printer_3: SPT1.1_printer03, shape=(5672, 5)
df_spt11_printer_4: SPT1.1_printer01, shape=(3625, 5)
SPT1.2 top 4 printers: ['SPT1.2_printer04', 'SPT1.2_printer02', 'SPT1.2_printer01', 'SPT1.2_printer03']
df_spt12_printer_1: SPT1.2_printer04, shape=(7309, 5)
df_spt12_printer_2: SPT1.2_printer02, shape=(7966, 5)
df_spt12_printer_3: SPT1.2_printer01, shape=(5183, 5)
df_spt12_printer_4: SPT1.2_printer03, shape=(5949, 5)


In [7]:
print(df_spt11_printer_4.head(5))

          date        printer_id  error_code               内容  occurance
5   2025-12-25  SPT1.1_printer01         589                         493
10  2025-12-25  SPT1.1_printer01         248  进片位卡尺异常或Mark点异常        138
24  2025-12-25  SPT1.1_printer01         337    出片称重位顶升气缸有异常片         60
27  2025-12-25  SPT1.1_printer01         592           等待上游有片         52
37  2025-12-25  SPT1.1_printer01         283            光幕被触发         35
